In [1]:
!pip install dash pandas plotly

In [2]:
import pandas as pd

df = pd.read_csv("DAV Assignment Task 2.csv")
print(df.columns)

Index(['Year', 'Company', 'Revenue (USD bn)', 'R&D Expenses (USD bn)',
       'Average Stock Price (USD)', 'Unnamed: 5'],
      dtype='object')


In [3]:
import pandas as pd
import dash
from dash import html, dcc, dash_table, Input, Output
import plotly.express as px

In [4]:
df = pd.read_csv("DAV Assignment Task 2.csv")

print(df.head())

   Year Company  Revenue (USD bn)  R&D Expenses (USD bn)  \
0  2020  Pfizer            41.908                  9.405   
1  2020     GSK            34.099                  5.098   
2  2021  Pfizer            81.288                 13.829   
3  2021     GSK            34.114                  5.278   
4  2022  Pfizer           100.330                 11.428   

   Average Stock Price (USD)  Unnamed: 5  
0                      36.50         NaN  
1                      31.15         NaN  
2                      45.10         NaN  
3                      32.17         NaN  
4                      50.20         NaN  


In [5]:
# KPI metrics
total_revenue = df["Revenue (USD bn)"].sum()
avg_stock = df["Average Stock Price (USD)"].mean()

# Advanced metric (high marks)
df["R&D Intensity"] = df["R&D Expenses (USD bn)"] / df["Revenue (USD bn)"]

In [6]:
# Create Dash app
app = dash.Dash(__name__)

# Layout
app.layout = html.Div([

    html.H1("Pharmaceutical Industry Dashboard", style={"textAlign": "center"}),

    html.P("Analysis of Revenue, R&D Expenses, and Stock Performance", 
           style={"textAlign": "center"}),

    # Dropdown filter
    html.Label("Select Company"),
    dcc.Dropdown(
        options=[
            {"label": "Pfizer", "value": "Pfizer"},
            {"label": "GSK", "value": "GSK"},
            {"label": "All", "value": "All"},
        ],
        value="All",
        id="company_filter"
    ),

    # KPI cards
    html.Div([

        html.Div([
            html.H4("Total Revenue"),
            html.P(f"{total_revenue:,.2f} USD bn")
        ], style={
            "padding": "20px",
            "backgroundColor": "#2E86C1",
            "color": "white",
            "width": "30%",
            "textAlign": "center",
            "borderRadius": "10px"
        }),

        html.Div([
            html.H4("Average Stock Price"),
            html.P(f"{avg_stock:,.2f} USD")
        ], style={
            "padding": "20px",
            "backgroundColor": "#28B463",
            "color": "white",
            "width": "30%",
            "textAlign": "center",
            "borderRadius": "10px"
        }),

    ], style={"display": "flex", "justifyContent": "space-around", "marginTop": "20px"}),

    # Charts
    dcc.Graph(id="revenue_chart"),
    dcc.Graph(id="rnd_chart"),
    dcc.Graph(id="stock_chart"),
    dcc.Graph(id="intensity_chart"),

    # Table
    html.H3("Dataset"),
    dash_table.DataTable(
        data=df.to_dict("records"),
        columns=[{"name": i, "id": i} for i in df.columns],
        page_size=10
    )

])

# Callback
@app.callback(
    Output("revenue_chart", "figure"),
    Output("rnd_chart", "figure"),
    Output("stock_chart", "figure"),
    Output("intensity_chart", "figure"),
    Input("company_filter", "value")
)
def update_dashboard(selected_company):

    if selected_company == "All":
        filtered_df = df
    else:
        filtered_df = df[df["Company"] == selected_company]

    fig1 = px.line(
        filtered_df,
        x="Year",
        y="Revenue (USD bn)",
        color="Company",
        markers=True,
        title="Revenue"
    )

    fig2 = px.bar(
        filtered_df,
        x="Year",
        y="R&D Expenses (USD bn)",
        color="Company",
        barmode="group",
        title="R&D Expenses"
    )

    fig3 = px.line(
        filtered_df,
        x="Year",
        y="Average Stock Price (USD)",
        color="Company",
        markers=True,
        title="Average Stock Price"
    )

    fig4 = px.line(
        filtered_df,
        x="Year",
        y="R&D Intensity",
        color="Company",
        markers=True,
        title="R&D Intensity (R&D / Revenue)"
    )

    return fig1, fig2, fig3, fig4


# Run app
if __name__ == "__main__":
    app.run(jupyter_mode="external", port=8052)

Dash app running on http://127.0.0.1:8052/
